# PA4 - PySpark (Team 6, CineMatch)
## Task 1. Define and prepare the data

**Project goal:** CineMatch recommends movies based on who worked on them, so we need a clean table of IMDb credits that later steps can group, join, and rank.

**Question 1:** Which crew roles show up in the top 10 billing spots of IMDb titles, and how high are they billed on average? (same as PA3)

**Question 2:** How does the mix of roles in the top 10 billing spots change across release eras?

**One row:** one credit (one person in one billing position on one title).
**Key:** (title_id, billing_order)

**Columns:** title_id (string, tt + digits), billing_order (int, 1 = first credited), person_id (string, nm + digits), role_category (string, job type), is_top10 (boolean, derived: billing_order <= 10).

In [20]:
!pip install -q pyspark

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import csv

spark = (SparkSession.builder
           .appName("CS131-PA4-Team6")
           .master("local[2]")
           .getOrCreate())

In [22]:
!git clone -q https://github.com/sjsu-cs131-f26/CineMatch-Netflix-Movies.git

BASE = "CineMatch-Netflix-Movies/pa4"
INPUT_PATH = f"{BASE}/data/sample/principals_sample.tsv"
PARQUET_PATH = f"{BASE}/work/prepared_credits.parquet"

checks = []
def record(check, value, note=""):
    checks.append([check, value, note])
    print(check, "=", value)

fatal: destination path 'CineMatch-Netflix-Movies' already exists and is not an empty directory.


In [23]:
schema = ("tconst STRING, ordering INT, nconst STRING, "
          "category STRING, job STRING, characters STRING")

raw = (spark.read
       .option("header", True)
       .option("sep", "\t")
       .schema(schema)
       .csv(INPUT_PATH))

raw.printSchema()
raw.show(5)
record("rows read", raw.count())

root
 |-- tconst: string (nullable = true)
 |-- ordering: integer (nullable = true)
 |-- nconst: string (nullable = true)
 |-- category: string (nullable = true)
 |-- job: string (nullable = true)
 |-- characters: string (nullable = true)

+---------+--------+---------+---------------+--------------------+----------+
|   tconst|ordering|   nconst|       category|                 job|characters|
+---------+--------+---------+---------------+--------------------+----------+
|tt0000001|       1|nm1588970|           self|                  \N|  ["Self"]|
|tt0000001|       2|nm0005690|       director|                  \N|        \N|
|tt0000001|       3|nm0005690|       producer|            producer|        \N|
|tt0000001|       4|nm0374658|cinematographer|director of photo...|        \N|
|tt0000002|       1|nm0721526|       director|                  \N|        \N|
+---------+--------+---------+---------------+--------------------+----------+
only showing top 5 rows
rows read = 1000


In [25]:
credits = raw.select(
    F.col("tconst").alias("title_id"),
    F.col("ordering").alias("billing_order"),
    F.col("nconst").alias("person_id"),
    F.col("category").alias("role_category"))

credits.printSchema()

root
 |-- title_id: string (nullable = true)
 |-- billing_order: integer (nullable = true)
 |-- person_id: string (nullable = true)
 |-- role_category: string (nullable = true)



In [27]:
missing = credits.filter(
    F.col("title_id").isNull() |
    F.col("person_id").isNull() |
    F.col("billing_order").isNull() |
    (F.col("billing_order") < 1) |
    F.col("role_category").isNull() |
    (F.col("role_category") == "\\N"))

record("rows with a missing or invalid required value", missing.count(),
       "excluded: a credit with no ID, position, or role can't be grouped or ranked")

valid = credits.filter(
    F.col("title_id").isNotNull() &
    F.col("person_id").isNotNull() &
    (F.col("billing_order") >= 1) &
    F.col("role_category").isNotNull() &
    (F.col("role_category") != "\\N"))

record("rows kept", valid.count())

rows with a missing or invalid required value = 0
rows kept = 1000


In [29]:
repeated = (valid.groupBy("title_id", "billing_order").count()
            .filter(F.col("count") > 1))

record("repeated (title_id, billing_order) keys", repeated.count(),
       "not removed: two rows with the same key could be different people")

repeated (title_id, billing_order) keys = 0


In [30]:
prepared = valid.withColumn("is_top10", F.col("billing_order") <= 10)

top10 = prepared.filter(F.col("is_top10")).count()
record("credits in the top 10 billing spots", top10)
prepared.show(5)

credits in the top 10 billing spots = 969
+---------+-------------+---------+---------------+--------+
| title_id|billing_order|person_id|  role_category|is_top10|
+---------+-------------+---------+---------------+--------+
|tt0000001|            1|nm1588970|           self|    true|
|tt0000001|            2|nm0005690|       director|    true|
|tt0000001|            3|nm0005690|       producer|    true|
|tt0000001|            4|nm0374658|cinematographer|    true|
|tt0000002|            1|nm0721526|       director|    true|
+---------+-------------+---------+---------------+--------+
only showing top 5 rows


In [31]:
prepared.write.mode("overwrite").parquet(PARQUET_PATH)
restored = spark.read.parquet(PARQUET_PATH)

restored.printSchema()
record("Parquet read-back rows", restored.count(), "should equal rows kept")
record("Parquet read-back top-10 credits", restored.filter(F.col("is_top10")).count(),
       "should equal credits in the top 10 billing spots")

root
 |-- title_id: string (nullable = true)
 |-- billing_order: integer (nullable = true)
 |-- person_id: string (nullable = true)
 |-- role_category: string (nullable = true)
 |-- is_top10: boolean (nullable = true)

Parquet read-back rows = 1000
Parquet read-back top-10 credits = 969


In [33]:
with open("quality_checks.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["check", "value", "note"])
    writer.writerows(checks)

from google.colab import files
files.download("quality_checks.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>